# Ellipsoid Fitting via Least Squares – Li & Griffiths (2004)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QL-UoHull/Ellipsoid-Fitting-via-Least-Squares-with-Ellipsoid-Specific-Constraints-Li-Griffiths-2004-/blob/main/notebooks/ellipsoid_fitting_demo.ipynb)

This notebook demonstrates the Python implementation of the **Li–Griffiths** least-squares ellipsoid-specific fitting algorithm.

**Reference:**  
Q. Li and J. G. Griffiths, *"Least Squares Ellipsoid Specific Fitting"*, Geometric Modeling and Processing, 2004.  
DOI: [10.1109/GMAP.2004.1290055](https://doi.org/10.1109/GMAP.2004.1290055)

---

## Contents
1. Installation
2. Algorithm overview
3. Example 1 – Unit sphere
4. Example 2 – Axis-aligned ellipsoid
5. Example 3 – Rotated & offset ellipsoid
6. Example 4 – Load point cloud from file
7. Visualisation
8. Accuracy benchmark

## 1. Installation

If running on **Google Colab**, uncomment and run the cell below:

In [ ]:
# Uncomment on Colab:
# !git clone https://github.com/QL-UoHull/Ellipsoid-Fitting-via-Least-Squares-with-Ellipsoid-Specific-Constraints-Li-Griffiths-2004-.git
# import sys; sys.path.insert(0, 'Ellipsoid-Fitting-via-Least-Squares-with-Ellipsoid-Specific-Constraints-Li-Griffiths-2004-/src')
# !pip install numpy scipy matplotlib -q

In [ ]:
import sys
import os

# For local development – add src/ to path
repo_root = os.path.join(os.path.dirname(os.path.abspath('.')), '')
src_path  = os.path.join(os.getcwd(), '..', 'src')
if os.path.isdir(src_path):
    sys.path.insert(0, os.path.abspath(src_path))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from ellipsoid_fitting import fit_ellipsoid
from ellipsoid_fitting.utils import generate_ellipsoid_points, load_point_cloud
from ellipsoid_fitting.visualization import plot_fit

print('Imports successful')

## 2. Algorithm overview

Any quadric surface can be written in algebraic form:

$$F(\mathbf{x}) = ax^2 + by^2 + cz^2 + 2fyz + 2gxz + 2hxy + 2px + 2qy + 2rz + d = 0$$

The 10-element coefficient vector is $\mathbf{v} = [a,b,c,f,g,h,p,q,r,d]^\top$.

**Li & Griffiths (2004)** derive an ellipsoid-specific constraint:

$$\kappa = 4ac - g^2 + 4bc - f^2 + 4ab - h^2 > 0$$

The fitting problem becomes a **constrained generalised eigenvalue** problem:

$$S\,\mathbf{v} = \lambda\,C\,\mathbf{v}$$

where $S = D^\top D$ is the scatter matrix and $C$ encodes the constraint $\kappa > 0$. The solution is the eigenvector corresponding to the **unique positive eigenvalue**.

## 3. Example 1 – Unit sphere

In [ ]:
rng = np.random.default_rng(42)

# Generate noisy points on a unit sphere
pts_sphere = generate_ellipsoid_points(
    center=[0, 0, 0],
    radii=[1, 1, 1],
    n_points=500,
    noise_std=0.02,
    rng=rng,
)

params_sphere = fit_ellipsoid(pts_sphere)
print(params_sphere)

print(f"\nError in centre: {np.linalg.norm(params_sphere.center):.4f}")
print(f"Error in radii:  {np.linalg.norm(params_sphere.radii - 1):.4f}")

## 4. Example 2 – Axis-aligned ellipsoid (a=3, b=2, c=1)

In [ ]:
TRUE_RADII = [3, 2, 1]

pts_ell = generate_ellipsoid_points(
    center=[0, 0, 0],
    radii=TRUE_RADII,
    n_points=600,
    noise_std=0.05,
    rng=rng,
)

params_ell = fit_ellipsoid(pts_ell)
print(params_ell)

print(f"\nTrue radii : {sorted(TRUE_RADII, reverse=True)}")
print(f"Fitted radii: {np.round(sorted(params_ell.radii, reverse=True), 3).tolist()}")

## 5. Example 3 – Rotated & offset ellipsoid

In [ ]:
# Build a rotation matrix (30° around Z, 15° around Y)
def Rz(deg):
    a = np.deg2rad(deg); c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def Ry(deg):
    a = np.deg2rad(deg); c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

R = Rz(30) @ Ry(15)
TRUE_CENTER = [5, -3, 2]
TRUE_RADII2  = [4, 2.5, 1.5]

pts_rot = generate_ellipsoid_points(
    center=TRUE_CENTER,
    radii=TRUE_RADII2,
    rotation=R,
    n_points=800,
    noise_std=0.05,
    rng=rng,
)

params_rot = fit_ellipsoid(pts_rot)
print(params_rot)

print(f"\nTrue  centre: {TRUE_CENTER}")
print(f"Fitted centre: {np.round(params_rot.center, 3).tolist()}")

## 6. Example 4 – Load point cloud from file

In [ ]:
data_file = os.path.join('..', 'data', 'ellipsoid_rotated_offset.csv')

if os.path.exists(data_file):
    pts_file = load_point_cloud(data_file)
    print(f'Loaded {len(pts_file)} points from file.')
    params_file = fit_ellipsoid(pts_file)
    print(params_file)
else:
    print('Data file not found – run from the notebooks/ directory, or adjust the path.')

## 7. Visualisation

In [ ]:
%matplotlib inline

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(131, projection='3d')
plot_fit(pts_sphere, params_sphere, title='Unit sphere', ax=ax1, show=False)

ax2 = fig.add_subplot(132, projection='3d')
plot_fit(pts_ell, params_ell, title='Axis-aligned (3,2,1)', ax=ax2, show=False)

ax3 = fig.add_subplot(133, projection='3d')
plot_fit(pts_rot, params_rot, title='Rotated & offset', ax=ax3, show=False)

plt.tight_layout()
plt.show()

## 8. Accuracy benchmark

Vary the noise level and measure centre/radii recovery error.

In [ ]:
noise_levels = [0.01, 0.02, 0.05, 0.10, 0.20]
true_r = np.array([3.0, 2.0, 1.0])

centre_errs, radii_errs = [], []

for sigma in noise_levels:
    pts = generate_ellipsoid_points([0,0,0], true_r, n_points=500,
                                    noise_std=sigma, rng=np.random.default_rng(sigma))
    p = fit_ellipsoid(pts)
    centre_errs.append(np.linalg.norm(p.center))
    radii_errs.append(np.linalg.norm(sorted(p.radii, reverse=True) - true_r))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(noise_levels, centre_errs, 'o-')
axes[0].set_xlabel('Noise σ'); axes[0].set_ylabel('Centre error'); axes[0].set_title('Centre recovery')
axes[1].plot(noise_levels, radii_errs, 's-', color='tomato')
axes[1].set_xlabel('Noise σ'); axes[1].set_ylabel('Radii error'); axes[1].set_title('Radii recovery')
plt.tight_layout(); plt.show()

print(f"{'Noise':>8} | {'Centre err':>12} | {'Radii err':>12}")
print('-' * 40)
for s, ce, re in zip(noise_levels, centre_errs, radii_errs):
    print(f"{s:>8.2f} | {ce:>12.4f} | {re:>12.4f}")